# Quantization Evaluation Analysis

This notebook tracks the quantization result for the GRPO study.

It now has two jobs:

1. Document the **0.5B quantization-floor result**: GRPO improves FP16 accuracy, but both bnb-NF4 and AWQ W4 erase the gain.
2. Inspect the **1.5B final matrix**: base vs GRPO under FP16, bnb-NF4 W4, and AWQ W4G128 on GSM8K test100.

The research question is:

> Does RL post-training change how a model quantizes?

The 0.5B result is a dev-loop negative/floor diagnostic. The 1.5B result is the first meaningful answer because the base model should remain functional after W4.

## How To Read This Notebook

The headline metrics are:

- `delta_fp16 = accuracy(g8_dr100_fp16) - accuracy(base_fp16)`
- `delta_w4 = accuracy(g8_dr100_w4) - accuracy(base_w4)`
- `delta_awq = accuracy(g8_dr100_awq) - accuracy(base_awq)`
- `gain_survival_w4 = delta_w4 - delta_fp16`
- `gain_survival_awq = delta_awq - delta_fp16`

Interpretation:

- `gain_survival_*` near zero: GRPO gain survives quantization.
- strongly negative `gain_survival_*`: quantization erased or reversed the GRPO gain.
- if base quantized accuracy is near floor, do not make the final research claim from that run.

## Configuration

Run this notebook on Great Lakes. It degrades gracefully: missing results are shown as `missing`, and the exact command to produce them is printed.

In [ ]:
from pathlib import Path
import json
import math

from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import pandas as pd

PQS_ROOT = Path('/scratch/huterer_root/huterer0/jiamingp/pqs')
REPO_ROOT = PQS_ROOT / 'repos/posttrain-quant-serve'
LOG_DIR = REPO_ROOT / 'logs'
OUTLIER_DIR_0P5 = PQS_ROOT / 'diagnostics/weight_outliers_0p5b_g8_dr100'

EVAL_SPECS = {
    '0p5_bnb_w4_test50': {
        'title': '0.5B bnb-NF4 W4 test50',
        'model_size': '0.5B',
        'role': 'dev floor diagnostic',
        'path': PQS_ROOT / 'evals/gsm8k_compare_test50_g8_dr100_bnb_w4_final',
        'job_id': '51161262',
        'command': '''PRECISIONS=fp16,w4 \
EVAL_SPLIT=test \
EVAL_LIMIT=50 \
EVAL_OUTPUT_DIR=/scratch/huterer_root/huterer0/jiamingp/pqs/evals/gsm8k_compare_test50_g8_dr100_bnb_w4_final \
sbatch --job-name=pqs-eval-w4-test50 \
  --account=cavestru0 \
  --time=00:15:00 \
  --export=ALL \
  slurm/eval_gsm8k_compare.sbatch''',
    },
    '0p5_awq_test50': {
        'title': '0.5B AWQ W4G128 test50',
        'model_size': '0.5B',
        'role': 'dev floor diagnostic',
        'path': PQS_ROOT / 'evals/gsm8k_compare_test50_g8_dr100_awq_w4g128',
        'job_id': '51163938',
        'command': '''PRECISIONS=fp16,awq \
EVAL_SPLIT=test \
EVAL_LIMIT=50 \
EVAL_OUTPUT_DIR=/scratch/huterer_root/huterer0/jiamingp/pqs/evals/gsm8k_compare_test50_g8_dr100_awq_w4g128 \
sbatch --job-name=pqs-eval-awq-test50 \
  --account=cavestru0 \
  --time=00:20:00 \
  --export=ALL \
  slurm/eval_gsm8k_compare.sbatch''',
    },
    '1p5_fp16_w4_awq_test100': {
        'title': '1.5B FP16 + bnb-W4 + AWQ test100',
        'model_size': '1.5B',
        'role': 'final result candidate',
        'path': PQS_ROOT / 'evals/gsm8k_compare_test100_qwen2_5_1_5b_g8_dr100_fp16_w4_awq',
        'job_id': '51173013',
        'command': '''MODEL_TAG=qwen2_5_1_5b \
PRECISIONS=fp16,w4,awq \
EVAL_SPLIT=test \
EVAL_LIMIT=100 \
EVAL_OUTPUT_DIR=/scratch/huterer_root/huterer0/jiamingp/pqs/evals/gsm8k_compare_test100_qwen2_5_1_5b_g8_dr100_fp16_w4_awq \
sbatch --job-name=pqs-eval-1p5-test100 \
  --account=cavestru0 \
  --time=00:45:00 \
  --export=ALL \
  slurm/eval_gsm8k_compare.sbatch''',
    },
}

ACTIVE_EVAL = '1p5_fp16_w4_awq_test100'

plt.style.use('default')
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.width', 180)

for key, spec in EVAL_SPECS.items():
    print(f"{key}: {spec['path']} exists={spec['path'].exists()}")

## Helper Functions

In [ ]:
def safe_display(df: pd.DataFrame | None, message: str = 'No rows to display.'):
    if df is None or df.empty:
        print(message)
    else:
        display(df)


def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.DataFrame(json.loads(line) for line in path.read_text().splitlines() if line.strip())


def read_matrix(eval_dir: Path) -> pd.DataFrame:
    path = eval_dir / 'results_matrix.csv'
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def fmt_pct(x):
    return None if pd.isna(x) else f'{100*x:.1f}%'


def metric(summary: dict, key: str):
    value = summary.get(key)
    if value is None:
        return None
    try:
        return float(value)
    except Exception:
        return value


def run_sacct_command(job_id: str) -> str:
    return f'sacct -j {job_id} --format=JobID,JobName%25,State,ExitCode,Elapsed'


def summarize_eval(label: str, spec: dict) -> dict:
    eval_dir = spec['path']
    summary = read_json(eval_dir / 'summary.json')
    matrix = read_matrix(eval_dir)
    row = {
        'eval': label,
        'title': spec['title'],
        'model_size': spec['model_size'],
        'role': spec['role'],
        'status': 'found' if summary and not matrix.empty else 'missing',
        'split': summary.get('split'),
        'limit': summary.get('limit'),
        'precisions': ','.join(summary.get('precisions', [])) if summary else None,
        'delta_fp16': metric(summary, 'delta_fp16'),
        'delta_w4': metric(summary, 'delta_w4'),
        'delta_awq': metric(summary, 'delta_awq'),
        'gain_survival_w4': metric(summary, 'gain_survival_w4'),
        'gain_survival_awq': metric(summary, 'gain_survival_awq'),
        'quant_drop_base_w4': metric(summary, 'quant_drop_base_w4'),
        'quant_drop_g8_w4': metric(summary, 'quant_drop_g8_w4'),
        'quant_drop_base_awq': metric(summary, 'quant_drop_base_awq'),
        'quant_drop_g8_awq': metric(summary, 'quant_drop_g8_awq'),
        'path': str(eval_dir),
        'job_id': spec.get('job_id'),
    }
    if not matrix.empty:
        for label_name in ['base_fp16', 'g8_dr100_fp16', 'base_w4', 'g8_dr100_w4', 'base_awq', 'g8_dr100_awq']:
            hit = matrix[matrix['label'] == label_name]
            row[f'{label_name}_acc'] = float(hit.iloc[0]['accuracy']) if len(hit) else None
    return row


def explain_gain_survival(summary: dict, quant: str) -> str:
    delta_fp16 = summary.get('delta_fp16')
    delta_q = summary.get(f'delta_{quant}')
    survival = summary.get(f'gain_survival_{quant}')
    if delta_fp16 is None or delta_q is None or survival is None:
        return f'Need both FP16 and {quant} rows.'
    if survival >= -0.05:
        return f'{quant}: GRPO gain mostly survives ({survival:+.3f}).'
    return f'{quant}: GRPO gain is erased/reversed ({survival:+.3f}).'


def print_missing_commands(label: str, spec: dict):
    print()
    print(f"Missing or incomplete: {label}")
    if spec.get('job_id'):
        print('Check job:')
        print(run_sacct_command(spec['job_id']))
    print('Run command:')
    print(spec['command'])

## 1. Artifact Checklist

This shows which eval directories are ready. If the 1.5B final matrix is missing, first check whether job `51173013` completed or timed out.

In [ ]:
artifact_rows = []
for label, spec in EVAL_SPECS.items():
    eval_dir = spec['path']
    files = {
        'summary_json': eval_dir / 'summary.json',
        'results_matrix': eval_dir / 'results_matrix.csv',
        'paired_quant': eval_dir / 'paired_quant_effect.csv',
    }
    row = {
        'eval': label,
        'title': spec['title'],
        'path': str(eval_dir),
        'job_id': spec.get('job_id'),
    }
    for name, path in files.items():
        row[name] = path.exists()
        row[f'{name}_kib'] = round(path.stat().st_size / 1024, 1) if path.exists() else None
    artifact_rows.append(row)

artifact_df = pd.DataFrame(artifact_rows)
safe_display(artifact_df)

for label, spec in EVAL_SPECS.items():
    summary_path = spec['path'] / 'summary.json'
    matrix_path = spec['path'] / 'results_matrix.csv'
    if not (summary_path.exists() and matrix_path.exists()):
        print_missing_commands(label, spec)

## 2. Cross-Run Scorecard

This is the compact comparison across the 0.5B floor diagnostics and the 1.5B final candidate.

In [ ]:
scorecard = pd.DataFrame([summarize_eval(label, spec) for label, spec in EVAL_SPECS.items()])
ordered_cols = [
    'eval', 'status', 'model_size', 'role', 'split', 'limit', 'precisions',
    'base_fp16_acc', 'g8_dr100_fp16_acc', 'base_w4_acc', 'g8_dr100_w4_acc', 'base_awq_acc', 'g8_dr100_awq_acc',
    'delta_fp16', 'delta_w4', 'delta_awq', 'gain_survival_w4', 'gain_survival_awq',
    'quant_drop_base_w4', 'quant_drop_g8_w4', 'quant_drop_base_awq', 'quant_drop_g8_awq',
    'path'
]
for col in ordered_cols:
    if col not in scorecard.columns:
        scorecard[col] = None
safe_display(scorecard[ordered_cols])

## 3. Active Eval Matrix

By default this loads the 1.5B final matrix. Change `ACTIVE_EVAL` in the configuration cell to inspect a different run.

In [ ]:
active_spec = EVAL_SPECS[ACTIVE_EVAL]
EVAL_DIR = active_spec['path']
SUMMARY_JSON = EVAL_DIR / 'summary.json'
RESULTS_MATRIX = EVAL_DIR / 'results_matrix.csv'
PAIRED_QUANT = EVAL_DIR / 'paired_quant_effect.csv'

summary = read_json(SUMMARY_JSON)
matrix = read_matrix(EVAL_DIR)

print('ACTIVE_EVAL =', ACTIVE_EVAL)
print('EVAL_DIR =', EVAL_DIR)
print('summary exists:', SUMMARY_JSON.exists())
print('matrix exists:', RESULTS_MATRIX.exists())

if not matrix.empty:
    matrix['accuracy_pct'] = matrix['accuracy'].map(fmt_pct)
    matrix['stderr_pct'] = matrix['accuracy_stderr'].map(fmt_pct)
    score_cols = [
        'variant', 'precision', 'label', 'correct', 'num_examples', 'accuracy', 'accuracy_stderr',
        'parse_rate', 'prompt_leak_rate', 'reference_ppl', 'completion_chars_mean', 'model'
    ]
    safe_display(matrix[score_cols])
else:
    print_missing_commands(ACTIVE_EVAL, active_spec)

headline_cols = ['delta_fp16', 'delta_w4', 'delta_awq', 'gain_survival_w4', 'gain_survival_awq', 'quant_drop_base_w4', 'quant_drop_g8_w4', 'quant_drop_base_awq', 'quant_drop_g8_awq']
headline = {col: summary.get(col) for col in headline_cols}
safe_display(pd.DataFrame([headline]) if summary else pd.DataFrame(), 'No summary metrics loaded.')

## 4. Headline Interpretation

This cell gives the plain-language answer for the active eval.

In [ ]:
if not summary or matrix.empty:
    display(Markdown('**No active result yet.** Check the job status or rerun the command printed above.'))
else:
    display(Markdown(f"### {active_spec['title']}"))
    delta_fp16 = summary.get('delta_fp16')
    display(Markdown(f"- FP16 GRPO gain: `{delta_fp16:+.3f}`" if delta_fp16 is not None else '- FP16 GRPO gain: missing'))
    for quant in ['w4', 'awq']:
        display(Markdown(f"- {explain_gain_survival(summary, quant)}"))

    if active_spec['model_size'] == '0.5B':
        display(Markdown(
            '**Interpretation:** this is a quantization-floor diagnostic. If quantized base accuracy is already near zero, '
            'the run is useful for debugging but not for the final research claim.'
        ))
    else:
        base_awq = scorecard.loc[scorecard['eval'] == ACTIVE_EVAL, 'base_awq_acc'].iloc[0] if 'base_awq_acc' in scorecard else None
        if pd.notna(base_awq) and base_awq < 0.15:
            display(Markdown(
                '**Warning:** even 1.5B base-AWQ is very low here. Treat the result cautiously and inspect predictions before writing the final claim.'
            ))
        else:
            display(Markdown(
                '**Interpretation:** if base quantized accuracy is meaningfully above floor, this run can answer whether GRPO survives W4/AWQ.'
            ))

## 5. Plots

The first plot compares raw accuracies. The second compares `delta` and `gain_survival`, which are the actual research quantities.

In [ ]:
found = scorecard[scorecard['status'] == 'found'].copy()
if found.empty:
    print('No completed evals to plot.')
else:
    acc_rows = []
    for _, row in found.iterrows():
        for label_name, col in [
            ('base_fp16', 'base_fp16_acc'), ('g8_fp16', 'g8_dr100_fp16_acc'),
            ('base_w4', 'base_w4_acc'), ('g8_w4', 'g8_dr100_w4_acc'),
            ('base_awq', 'base_awq_acc'), ('g8_awq', 'g8_dr100_awq_acc')
        ]:
            if pd.notna(row.get(col)):
                acc_rows.append({'eval': row['eval'], 'model_size': row['model_size'], 'label': label_name, 'accuracy': row[col]})
    acc_df = pd.DataFrame(acc_rows)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    if not acc_df.empty:
        pivot = acc_df.pivot(index='label', columns='eval', values='accuracy')
        pivot.plot(kind='bar', ax=axes[0])
        axes[0].set_title('Accuracy by run and precision')
        axes[0].set_ylabel('accuracy')
        axes[0].grid(axis='y', alpha=0.25)
        axes[0].tick_params(axis='x', rotation=30)

    metric_cols = ['delta_fp16', 'delta_w4', 'delta_awq', 'gain_survival_w4', 'gain_survival_awq']
    metric_df = found.set_index('eval')[metric_cols]
    metric_df.plot(kind='bar', ax=axes[1])
    axes[1].axhline(0, color='black', linewidth=1)
    axes[1].set_title('GRPO gain and gain survival')
    axes[1].set_ylabel('accuracy delta')
    axes[1].grid(axis='y', alpha=0.25)
    axes[1].tick_params(axis='x', rotation=30)

    fig.tight_layout()
    plt.show()

## 6. Paired Quantization Effects

This shows how many individual GSM8K examples changed when the same checkpoint moved from FP16 to W4 or AWQ.

In [ ]:
paired = pd.read_csv(PAIRED_QUANT) if PAIRED_QUANT.exists() else pd.DataFrame()
safe_display(paired.head(), 'No paired quantization file loaded for active eval.')

if not paired.empty:
    group_cols = ['variant', 'quant_precision', 'quant_change'] if 'quant_precision' in paired.columns else ['variant', 'quant_change']
    counts = paired.groupby(group_cols).size().reset_index(name='count')
    safe_display(counts)

    if 'quant_precision' in paired.columns:
        pivot = counts.pivot_table(index=['variant', 'quant_precision'], columns='quant_change', values='count', fill_value=0)
    else:
        pivot = counts.pivot_table(index='variant', columns='quant_change', values='count', fill_value=0)
    for col in ['improved', 'worsened', 'unchanged']:
        if col not in pivot.columns:
            pivot[col] = 0
    safe_display(pivot[['improved', 'worsened', 'unchanged']])

    ax = pivot[['improved', 'worsened']].plot(kind='bar', figsize=(8, 4), color=['tab:green', 'tab:red'])
    ax.set_title('Examples changed by quantization')
    ax.set_ylabel('count')
    ax.grid(axis='y', alpha=0.25)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 7. Inspect Changed Examples

Use this to see whether quantization changes are real answer changes or parser/format quirks.

In [ ]:
if paired.empty:
    print('No paired rows to inspect.')
else:
    changed = paired[paired['quant_change'] != 'unchanged'].copy()
    inspect_cols = [
        'variant', 'quant_precision', 'index', 'target_answer', 'fp16_answer', 'quant_answer',
        'fp16_correct', 'quant_correct', 'quant_change', 'question'
    ]
    inspect_cols = [col for col in inspect_cols if col in changed.columns]
    safe_display(changed[inspect_cols].sort_values([col for col in ['variant', 'quant_precision', 'quant_change', 'index'] if col in changed.columns]))

## 8. Prediction Drilldown

Set `EXAMPLE_INDEX` to inspect all available completions for one GSM8K item.

In [ ]:
EXAMPLE_INDEX = 1

if matrix.empty:
    print('No active matrix loaded.')
else:
    prediction_labels = matrix['label'].tolist()
    prediction_frames = {label: read_jsonl(EVAL_DIR / f'{label}_predictions.jsonl') for label in prediction_labels}

    for label, frame in prediction_frames.items():
        if frame.empty or 'index' not in frame.columns or EXAMPLE_INDEX not in set(frame['index']):
            continue
        row = frame[frame['index'] == EXAMPLE_INDEX].iloc[0]
        display(Markdown(f"### {label}"))
        print('target:', row.get('target_answer'))
        print('parsed:', row.get('parsed_answer'))
        print('correct:', row.get('exact_match'))
        print('prompt_leak:', row.get('prompt_leak'))
        print()
        print('question:')
        print(row.get('question'))
        print()
        print('completion:')
        print(row.get('completion'))

## 9. Weight Diagnostics Context

The earlier 0.5B weight/outlier diagnostic found near-identical W4 proxy error between base and GRPO. That supports the idea that the 0.5B failure is behavioral/fragility rather than an obvious global weight-distribution shift.

In [ ]:
weight_summary_path = OUTLIER_DIR_0P5 / 'weight_outlier_summary.json'
weight_summary = read_json(weight_summary_path)

weight_rows = []
for model_summary in weight_summary.get('models', []):
    weight_rows.append({
        'model': model_summary.get('model'),
        'num_weight_tensors': model_summary.get('num_weight_tensors'),
        'channel_outlier_frac_weighted': model_summary.get('channel_outlier_frac_weighted'),
        'abs_outlier_frac_weighted': model_summary.get('abs_outlier_frac_weighted'),
        'w4_relative_mse_weighted': model_summary.get('w4_relative_mse_weighted'),
        'w4_snr_db_mean': model_summary.get('w4_snr_db_mean'),
    })
safe_display(pd.DataFrame(weight_rows), 'No 0.5B weight diagnostic summary loaded.')

## 10. Current Conclusion

Use this section after the 1.5B final matrix finishes. The notebook deliberately does not invent numbers: if `summary.json` is missing, it prints the command to run instead.

In [ ]:
final_label = '1p5_fp16_w4_awq_test100'
final_row = scorecard[scorecard['eval'] == final_label]

if final_row.empty or final_row.iloc[0]['status'] != 'found':
    display(Markdown('**Final 1.5B matrix is not loaded yet.** Run or check the job printed in the artifact checklist.'))
else:
    r = final_row.iloc[0]
    display(Markdown(
        f"**1.5B result:** FP16 gain `{r['delta_fp16']:+.3f}`, "
        f"bnb-W4 gain `{r['delta_w4']:+.3f}`, AWQ gain `{r['delta_awq']:+.3f}`. "
        f"Gain survival: bnb-W4 `{r['gain_survival_w4']:+.3f}`, AWQ `{r['gain_survival_awq']:+.3f}`."
    ))

    if pd.notna(r['gain_survival_awq']) and r['gain_survival_awq'] >= -0.05:
        display(Markdown('**Claim direction:** the GRPO gain mostly survives AWQ W4 on 1.5B.'))
    elif pd.notna(r['gain_survival_awq']):
        display(Markdown('**Claim direction:** AWQ W4 substantially erases/reverses the GRPO gain on 1.5B. Inspect examples before writing the final result.'))
    else:
        display(Markdown('**Claim direction:** AWQ rows missing; cannot answer the main question yet.'))